# Data Cleaning and Feature Generation

This notebook shows how the raw ChemFluor dataset is cleaned and converted into machine learning features.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)

## Load and Clean the Dataset

The training dataset should be stored in the project root as `chemfluor_data.csv`.

Required columns:

- `SMILES`
- `solvent`
- `Emission/nm`
- `PLQY`

In [ ]:
from src.data import load_raw_data, clean_data

raw = load_raw_data()
print("Raw shape:", raw.shape)
raw.head()

In [ ]:
cleaned, stats = clean_data(raw)
print("Cleaned shape:", cleaned.shape)
stats

In [ ]:
cleaned[["SMILES", "canonical_smiles", "solvent", "Emission/nm", "PLQY"]].head()

## Build the Feature Matrix

This step creates Morgan fingerprints, MACCS keys, RDKit descriptors, one-hot solvent features, and solvent descriptor features if `solvent_descriptors.csv` is available.

In [ ]:
from src.features import build_feature_matrix

X = build_feature_matrix(cleaned)
print("Feature matrix shape:", X.shape)
X.head()

In [ ]:
print("Number of Morgan features:", sum(c.startswith("morgan_") for c in X.columns))
print("Number of MACCS features:", sum(c.startswith("maccs_") for c in X.columns))
print("Number of solvent one-hot features:", sum(c.startswith("solvent_") for c in X.columns))
print("Number of solvent descriptor features:", sum(c.startswith("solvdesc_") for c in X.columns))

## Notes

If the solvent descriptor file is blank or missing, the model will still run using one-hot solvent features only. If descriptor values are partially missing, they are reported and median-imputed.